# LLM Routing System: Model Training

We use **Sentence Embeddings** to represent queries and a **Logistic Regression** classifier to predict the best target model.

## 1. Setup
Install dependencies and import libraries.

In [ ]:
!pip install -q sentence-transformers scikit-learn pandas tqdm joblib

In [ ]:
import json
import os
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from tqdm.auto import tqdm
import joblib

# 1. Download/Verify Data
if not os.path.exists('data.jsonl'):
    print("Downloading dataset...")
    !wget -q https://raw.githubusercontent.com/kutt27/llm-routing/master/Data/data.jsonl

# 2. Load Data into DataFrame
queries = []
labels = []

with open('data.jsonl', 'r') as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith('//'):
            continue
        try:
            obj = json.loads(line)
            queries.append(obj['query'])
            labels.append(obj['label'])
        except json.JSONDecodeError:
            continue

df = pd.DataFrame({'query': queries, 'label': labels})
print(f"Loaded {len(df)} samples.")
df.head()

## 2. Generate Embeddings
Using `all-MiniLM-L6-v2` to convert text into numerical vectors.

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print("Encoding queries...")
X = embedder.encode(df['query'].tolist(), show_progress_bar=True)
y = df['label'].values

## 3. Train Classifier
Training a Logistic Regression model to select the best model name.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = LogisticRegression(max_iter=1000, multi_class='multinomial')
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## 4. Test the Router

In [ ]:
def route(query):
    emb = embedder.encode([query])
    return clf.predict(emb)[0]

samples = [
    "How do I centers a div in CSS?",
    "Explain the theory of general relativity.",
    "What is 2+2?"
]

for s in samples:
    print(f"Query: {s} -> Route: {route(s)}")

## 5. Export Model

In [ ]:
os.makedirs('models', exist_ok=True)
joblib.dump(clf, 'models/router_clf.joblib')
print("Saved classifier to models/router_clf.joblib")